# 01 — Top-down pipeline only

Questo notebook esegue **solo la prima parte/top-down** della pipeline GORE. La generazione di description, actors, HLG e LLG usa i generatori originali; non viene eseguito il ciclo bottom-up.

Configurazione sperimentale: **Actors, HLG e LLG Zero-shot con critic Few-shot abilitato**. Il critic usa quattro esempi, soglia `8.5/10` e al massimo tre iterazioni. Gli LLG vengono generati e valutati globalmente a partire dall'intero insieme degli HLG.

Gli output strutturati vengono salvati in `output/top_down/` e possono essere riutilizzati dal notebook bottom-up e dal notebook di valutazione.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_DIR = PROJECT_ROOT / "notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print("Project root:", PROJECT_ROOT)

from key import get_key_llama, count_Llama_keys

groq_key = get_key_llama()

print("Chiave Groq per Qwen presente:", bool(groq_key))
print("Formato Groq plausibile:", groq_key.startswith("gsk_") if groq_key else False)
print("Numero chiavi Groq:", count_Llama_keys())
print("LM Studio deve essere in ascolto su http://127.0.0.1:1234/v1")

Project root: C:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement
Chiave Groq per Qwen presente: True
Formato Groq plausibile: True
Numero chiavi Groq: 5
LM Studio deve essere in ascolto su http://127.0.0.1:1234/v1


## 1. Configurazione

La documentazione segue il percorso originale: se la ground truth contiene `link-readme`, viene chiamato `generate_description(...)`; altrimenti viene usata direttamente la descrizione disponibile nella ground truth. Non viene introdotto chunking aggiuntivo nella baseline.


In [2]:
import json

from groundtruth import ALL_GROUNDTRUTHS
from src.extraction.extractor import (
    generate_description,
    generate_actors,
    generate_high_level_goals,
    generate_low_level_goals,
    DESCRIPTION_SOURCE_MAX_CHARS,
)
from src.self_critique.refine_response import (
    EvalMode,
    generate_response_with_reflection,
)
import src.self_critique.refine_response as reflection
from src.examples.shot_learning import ShotPromptingMode
from src.utils import get_markdown
import src.llm_clients as llm_clients

GROUNDTRUTHS = list(ALL_GROUNDTRUTHS)
if not GROUNDTRUTHS:
    raise RuntimeError("No projects found in groundtruth.py")

ACTOR_PROMPTING_MODE = ShotPromptingMode.ZERO_SHOT
HLG_PROMPTING_MODE = ShotPromptingMode.ZERO_SHOT
LLG_PROMPTING_MODE = ShotPromptingMode.ZERO_SHOT
ACTOR_EVALUATOR_ABLATION = False
HLG_EVALUATOR_ABLATION = False
LLG_EVALUATOR_ABLATION = False
TOP_DOWN_MAX_ATTEMPTS = 3
TOP_DOWN_QUALITY_THRESHOLD = 8.5
TOP_DOWN_EVALUATOR_FEW_SHOT_EXAMPLE_COUNT = 4
PIPELINE_PROFILE = "ALL_ZERO_SHOT_WITH_FEW_SHOT_CRITIC"
# Keep the established filename so a rerun replaces the stale baseline
# instead of creating duplicate datasets for the bottom-up notebook.
OUTPUT_FILE_PROFILE = "ZERO_SHOT_feedback_original_top_down"

OUTPUT_ROOT = PROJECT_ROOT / "output"
TOP_DOWN_DIR = OUTPUT_ROOT / "top_down"
TOP_DOWN_DIR.mkdir(parents=True, exist_ok=True)

def top_down_output_file_for(dataset_name: str) -> Path:
    return TOP_DOWN_DIR / f"{dataset_name}_{OUTPUT_FILE_PROFILE}_top_down.json"

def run_top_down_reflection(**kwargs):
    """Run the paper loop parameters without changing bottom-up defaults."""
    previous_threshold = reflection.QUALITY_THRESHOLD
    reflection.QUALITY_THRESHOLD = TOP_DOWN_QUALITY_THRESHOLD
    try:
        return generate_response_with_reflection(
            max_attempts=TOP_DOWN_MAX_ATTEMPTS,
            **kwargs,
        )
    finally:
        reflection.QUALITY_THRESHOLD = previous_threshold

print(f"Ground-truth projects discovered: {len(GROUNDTRUTHS)}")
for project_index, project in enumerate(GROUNDTRUTHS, start=1):
    print(f"  {project_index}/{len(GROUNDTRUTHS)} - {project['name']}")
print("Top-down output directory:", TOP_DOWN_DIR)
print("Qwen generator (Groq):", llm_clients.QWEN_GENERATOR_MODEL)
print("Qwen generator max output tokens:", llm_clients.QWEN_GENERATOR_MAX_TOKENS)
print("Ministral critic (LM Studio):", llm_clients.configured_critic_model())
print("LM Studio endpoint:", llm_clients.LM_STUDIO_BASE_URL)
print("Ministral critic max output tokens:", llm_clients.MINISTRAL_CRITIC_MAX_TOKENS)
print("Actor prompting:", ACTOR_PROMPTING_MODE.name, "| critic: ON (Few-shot)")
print("HLG prompting:", HLG_PROMPTING_MODE.name, "| critic: ON (Few-shot)")
print("LLG prompting:", LLG_PROMPTING_MODE.name, "| critic: ON (Few-shot)")
print("Top-down evaluator few-shot examples:", TOP_DOWN_EVALUATOR_FEW_SHOT_EXAMPLE_COUNT)
print("Top-down evaluator description character budget:", reflection.TOP_DOWN_EVALUATOR_DESCRIPTION_MAX_CHARS)
print("Top-down quality threshold:", TOP_DOWN_QUALITY_THRESHOLD)
print("Top-down max reflection attempts:", TOP_DOWN_MAX_ATTEMPTS)


Ground-truth projects discovered: 12
  1/12 - SIA Project 25 26
  2/12 - Assegno Unico Universale - SIA Project 24 25
  3/12 - La Reine Marlene - SIA Project 23 24
  4/12 - Ethical Purchasing Group - SIA Project 22 23
  5/12 - Event Organization Portal - SIA Project 21 22
  6/12 - Calculator
  7/12 - Fitness Tracker
  8/12 - Genome Nexus
  9/12 - Gestao Hospital
  10/12 - London Ambulance Service
  11/12 - Planning Poker
  12/12 - Recycling
Top-down output directory: C:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\top_down
Qwen generator (Groq): qwen/qwen3.8-27b
Qwen generator max output tokens: 4000
Ministral critic (LM Studio): mistralai/ministral-3-14b-reasoning
LM Studio endpoint: http://127.0.0.1:1234/v1
Ministral critic max output tokens: 700
Actor prompting: ZERO_SHOT | critic: ON (Few-shot)
HLG prompting: ZERO_SHOT | critic: ON (Few-shot)
LLG prompting: ZERO_SHOT | critic: ON (Few-shot)
Top-down evaluator few-shot examples: 4
Top-down 

## 2. Esecuzione top-down e salvataggio

Per ogni dataset viene costruita la `description` con il percorso originale. Actors, HLG e LLG partono da una generazione Zero-shot e vengono valutati dal critic Few-shot. Come nella pipeline originale, tutti gli HLG e tutti gli LLG vengono valutati globalmente. Il JSON conserva anche gli oggetti Pydantic strutturati necessari al successivo bottom-up. Gli output già visualizzati appartengono all'esecuzione precedente e vengono sostituiti rieseguendo le celle.


In [3]:
generated_descriptions = {}
generated_actor_objects = {}
generated_hl_objects = {}
generated_ll_objects = {}
baseline_errors = {}


def run_top_down_for_dataset(groundtruth: dict) -> None:
    dataset_name = groundtruth["name"]

    try:
        # ---------------------------------------------------------------
        # 1. ORIGINAL documentation preprocessing
        # ---------------------------------------------------------------
        if "link-readme" in groundtruth:
            documentation_link = groundtruth["link-readme"]

            # Saved only for traceability; generate_description() below follows
            # the original repository path and reads the README itself.
            raw_description = get_markdown(documentation_link)
            fallback_description = str(groundtruth.get("description", "")).strip()
            source_is_oversized_html = (
                len(raw_description) > DESCRIPTION_SOURCE_MAX_CHARS * 2
                and "<html" in raw_description[:2000].lower()
            )
            if source_is_oversized_html and fallback_description:
                description = fallback_description
                preprocessing_info = {
                    "chunked": False,
                    "chunkCount": 1,
                    "originalChars": len(raw_description),
                    "processedChars": len(description),
                    "strategy": "groundtruth_description_fallback_for_html",
                    "sourceMaxChars": DESCRIPTION_SOURCE_MAX_CHARS,
                }
            else:
                generated_description = generate_description(
                    documentation_link, documentation_text=raw_description
                )
                description = generated_description.description.strip()
                preprocessing_info = {
                    "chunked": False,
                    "chunkCount": 1,
                    "originalChars": len(raw_description),
                    "processedChars": len(description),
                    "strategy": "bounded_generate_description",
                    "sourceMaxChars": DESCRIPTION_SOURCE_MAX_CHARS,
                }
        else:
            raw_description = str(groundtruth["description"])
            description = raw_description

            preprocessing_info = {
                "chunked": False,
                "chunkCount": 1,
                "originalChars": len(raw_description),
                "processedChars": len(description),
                "strategy": "groundtruth_description",
            }

        print(
            f"[{dataset_name}] Documentation ready | "
            f"raw={len(raw_description)} chars | "
            f"pipeline={len(description)} chars"
        )

        # ---------------------------------------------------------------
        # 2. ACTORS: Zero-shot generation + Few-shot critic/reflection
        # ---------------------------------------------------------------
        actors, actors_score, actors_critique = run_top_down_reflection(
            target_type="Actors",
            call_function=generate_actors,
            define_args=(description,),
            eval_mode=EvalMode.ACTORS,
            eval_args=(description,),
            shotPromptingMode=ACTOR_PROMPTING_MODE,
            llama_ablation=ACTOR_EVALUATOR_ABLATION,
        )

        # ---------------------------------------------------------------
        # 3. HLG: Zero-shot generation + Few-shot critic/reflection
        # ---------------------------------------------------------------
        high_level_goals, hl_score, hl_critique = (
            run_top_down_reflection(
                target_type="High Level Goals",
                call_function=generate_high_level_goals,
                define_args=(description, actors),
                eval_mode=EvalMode.HIGH_LEVEL,
                eval_args=(description, actors),
                shotPromptingMode=HLG_PROMPTING_MODE,
                llama_ablation=HLG_EVALUATOR_ABLATION,
            )
        )

        # ---------------------------------------------------------------
        # 4. LLG: global Zero-shot generation + Few-shot critic/reflection
        # ---------------------------------------------------------------
        low_level_goals, ll_score, ll_critique = run_top_down_reflection(
            target_type="Low Level Goals",
            call_function=generate_low_level_goals,
            define_args=(high_level_goals,),
            eval_mode=EvalMode.LOW_LEVEL,
            eval_args=(description, actors, high_level_goals),
            shotPromptingMode=LLG_PROMPTING_MODE,
            llama_ablation=LLG_EVALUATOR_ABLATION,
        )
        if not actors.actors or not high_level_goals.goals or not low_level_goals.low_level_goals:
            raise RuntimeError(
                "Top-down extraction produced an empty actor or goal collection; "
                "the previous output file was not overwritten."
            )

        generated_descriptions[dataset_name] = description
        generated_actor_objects[dataset_name] = actors
        generated_hl_objects[dataset_name] = high_level_goals
        generated_ll_objects[dataset_name] = low_level_goals

        output = {
            "name": dataset_name,
            "description": description,
            "rawDescription": raw_description,
            "documentationPreprocessing": preprocessing_info,
            "actors": [actor.name for actor in actors.actors],
            "highLevelGoals": [
                goal.description for goal in high_level_goals.goals
            ],
            "lowLevelGoals": [
                goal.description for goal in low_level_goals.low_level_goals
            ],
            "structuredActors": actors.model_dump(mode="json"),
            "structuredHighLevelGoals": high_level_goals.model_dump(mode="json"),
            "structuredLowLevelGoals": low_level_goals.model_dump(mode="json"),
            "baselineScores": {
                "actors": actors_score,
                "highLevelGoals": hl_score,
                "lowLevelGoals": ll_score,
            },
            "baselineCritiques": {
                "actors": actors_critique,
                "highLevelGoals": hl_critique,
                "lowLevelGoals": ll_critique,
            },
            "baselineConfiguration": {
                "implementation": "original top-down prompts with token-safe compact target serialization; Qwen generator and local Ministral critic",
                "generatorModel": llm_clients.QWEN_GENERATOR_MODEL,
                "actorPromptingMode": ACTOR_PROMPTING_MODE.name,
                "actorEvaluatorEnabled": not ACTOR_EVALUATOR_ABLATION,
                "highLevelGoalPromptingMode": HLG_PROMPTING_MODE.name,
                "lowLevelGoalPromptingMode": LLG_PROMPTING_MODE.name,
                "highLevelGoalEvaluatorEnabled": not HLG_EVALUATOR_ABLATION,
                "lowLevelGoalEvaluatorEnabled": not LLG_EVALUATOR_ABLATION,
                "goalEvaluatorModel": llm_clients.configured_critic_model(),
                "qualityThreshold": TOP_DOWN_QUALITY_THRESHOLD,
                "maxReflectionAttempts": TOP_DOWN_MAX_ATTEMPTS,
                "evaluatorFewShotExampleCount": TOP_DOWN_EVALUATOR_FEW_SHOT_EXAMPLE_COUNT,
                "evaluatorPromptStrategy": "original_prompt_with_compact_target_serialization",
                "evaluatorMaxInputChars": reflection.TOP_DOWN_EVALUATOR_MAX_INPUT_CHARS,
                "evaluatorDescriptionMaxChars": reflection.TOP_DOWN_EVALUATOR_DESCRIPTION_MAX_CHARS,
            },
            "pipelineConfiguration": {
                "actors": {
                    "promptingMode": ACTOR_PROMPTING_MODE.name,
                    "evaluator": not ACTOR_EVALUATOR_ABLATION,
                },
                "highLevelGoals": {
                    "promptingMode": HLG_PROMPTING_MODE.name,
                    "evaluator": not HLG_EVALUATOR_ABLATION,
                },
                "lowLevelGoals": {
                    "promptingMode": LLG_PROMPTING_MODE.name,
                    "evaluator": not LLG_EVALUATOR_ABLATION,
                    "generationStrategy": "all_hlgs_in_one_call",
                    "evaluationStrategy": "single_global_call_with_original_prompt",
                },
                "profile": PIPELINE_PROFILE,
            },
        }

        output_file = top_down_output_file_for(dataset_name)
        with output_file.open("w", encoding="utf-8") as file:
            json.dump(
                output,
                file,
                indent=4,
                ensure_ascii=False,
                default=str,
            )

        print(
            f"[{dataset_name}] saved: {output_file.name} | "
            f"Actors={len(actors.actors)} (Zero-shot, evaluator ON Few-shot) | "
            f"HLG={len(high_level_goals.goals)} (Zero-shot, evaluator ON Few-shot) | "
            f"LLG={len(low_level_goals.low_level_goals)} "
            f"(Zero-shot, evaluator ON Few-shot, single global call) | "
            f"evaluator={llm_clients.configured_critic_model()}"
        )

    except Exception as error:
        error_text = " ".join(str(error).split())
        if len(error_text) > 800:
            error_text = error_text[:797] + "..."
        baseline_errors[dataset_name] = (
            f"{type(error).__name__}: {error_text}"
        )
        print(
            f"[{dataset_name}] Top-down extraction failed: "
            f"{type(error).__name__}: {error_text}"
        )


for project_index, groundtruth in enumerate(GROUNDTRUTHS, start=1):
    print(
        f"\n=== PROJECT {project_index}/{len(GROUNDTRUTHS)}: "
        f"{groundtruth['name']} ==="
    )
    run_top_down_for_dataset(groundtruth)



=== PROJECT 1/12: SIA Project 25 26 ===
[SIA Project 25 26] Documentation ready | raw=5824 chars | pipeline=5824 chars
Actors STARTING... (attempt 1)
No feedback provided!
Actors DONE...
Actors: 6 actors
Evaluation for Actors STARTING... (format attempt 1)
Evaluation for Actors DONE...
Score: 6.0
Critique: The actors list is mostly representative of the system's description but includes some inconsistencies. "External Maintenance Personnel" is not clearly described in the software description as an actor with direct interaction or responsibility within the Participium system. Additionally, while "Organization Office" and "Technical Office" are mentioned, their roles overlap significantly with what would typically be considered administrative functions (e.g., reviewing, approving, rejecting reports). The "Administrator" role is also somewhat redundant with these offices unless further distinguished. Clarifying or consolidating these roles would improve accuracy.
Unsatisfactory score. R

## 3. Riepilogo

Al termine viene mostrato il numero di dataset completati e falliti. Ogni dataset completato produce un JSON separato in `output/top_down/`.


In [4]:
completed_outputs = globals().get("generated_ll_objects", {})
execution_errors = globals().get("baseline_errors", {})
output_directory = globals().get("TOP_DOWN_DIR")

print("\nTop-down execution summary")
print("--------------------------")
print(f"Configured datasets: {len(GROUNDTRUTHS)}")
print(f"Completed datasets: {len(completed_outputs)}")
print(f"Failed datasets: {len(execution_errors)}")
print(f"Output directory: {output_directory or 'pipeline not executed yet'}")

if execution_errors:
    print("Errors:")
    for dataset_name, error in execution_errors.items():
        print(f"- {dataset_name}: {error}")
elif not completed_outputs:
    print("Pipeline not executed: run all previous cells first.")


Top-down execution summary
--------------------------
Configured datasets: 12
Completed datasets: 12
Failed datasets: 0
Output directory: C:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\top_down
